In [1]:
import kagglehub, shutil

path = kagglehub.dataset_download(
    "ealtman2019/ibm-transactions-for-anti-money-laundering-aml",
    force_download=True,
)
shutil.copytree(path, "/workspace/money_laundry_ai_/datasets", dirs_exist_ok=True)

Extracting files...


'/workspace/money_laundry_ai_/datasets'

In [4]:
import sys, subprocess, shutil, glob
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", "kagglehub"],
                   capture_output=True, text=True)
print("pip exit:", r.returncode); print(r.stderr[-2000:])


pip exit: 0



In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("berkanoztas/synthetic-transaction-monitoring-dataset-aml", force_download=True,)

shutil.copytree(path, "/workspace/money_laundry_ai_/datasets/SAML-D", dirs_exist_ok=True)

100%|██████████| 193M/193M [00:07<00:00, 28.7MB/s] 

Extracting files...


'/workspace/money_laundry_ai_/datasets/SAML-D'

In [6]:
# SAML-D 프로파일 — 스키마, 건수, 기간, 값 종류, 라벨 분포
import duckdb, pandas as pd
from IPython.display import display

CSV = "/workspace/money_laundry_ai_/datasets/SAML-D/SAML-D.csv"   # 받은 위치로 바꾸기
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE VIEW s AS SELECT * FROM read_csv_auto('{CSV}', header=true)")
display(con.execute("DESCRIBE s").df())                                          # 컬럼 이름·타입

display(con.execute("""
    SELECT count(*) AS 거래, sum(Is_laundering) AS 세탁, 100 * avg(Is_laundering) AS 세탁_비율_pct,
           min(Date) AS 시작일, max(Date) AS 종료일, count(DISTINCT Date) AS 날짜_수,
           count(DISTINCT Sender_account) AS 송신_계좌, count(DISTINCT Receiver_account) AS 수신_계좌,
           min(Amount) AS 최소_금액, median(Amount) AS 중앙_금액, max(Amount) AS 최대_금액
    FROM s
""").df().T)

for c in ["Payment_type", "Payment_currency", "Received_currency", "Sender_bank_location", "Receiver_bank_location"]:
    print(f"── {c}")
    display(con.execute(f"""SELECT {c}, count(*) AS 거래, sum(Is_laundering) AS 세탁,
                                   round(100 * avg(Is_laundering), 4) AS 세탁_비율_pct
                            FROM s GROUP BY 1 ORDER BY 2 DESC""").df())

print("── Laundering_type (정상·의심 유형별)")
display(con.execute("""SELECT Is_laundering, Laundering_type, count(*) AS 거래
                       FROM s GROUP BY ALL ORDER BY Is_laundering DESC, 거래 DESC""").df())

print("── 월별 거래·세탁")
display(con.execute("""SELECT strftime(CAST(Date AS DATE), '%Y-%m') AS 월, count(*) AS 거래, sum(Is_laundering) AS 세탁
                       FROM s GROUP BY 1 ORDER BY 1""").df())
display(con.execute("SELECT * FROM s LIMIT 5").df())

,column_name,column_type,null,key,default,extra
0,Time,TIME,YES,None,None,None
1,Date,DATE,YES,None,None,None
2,Sender_account,BIGINT,YES,None,None,None
3,Receiver_account,BIGINT,YES,None,None,None
4,Amount,DOUBLE,YES,None,None,None
5,Payment_currency,VARCHAR,YES,None,None,None
6,Received_currency,VARCHAR,YES,None,None,None
7,Sender_bank_location,VARCHAR,YES,None,None,None
8,Receiver_bank_location,VARCHAR,YES,None,None,None
9,Payment_type,VARCHAR,YES,None,None,None


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,0
거래,9504852
세탁,9873.0
세탁_비율_pct,0.103873
시작일,2022-10-07 00:00:00
종료일,2023-08-23 00:00:00
날짜_수,321
송신_계좌,292715
수신_계좌,652266
최소_금액,3.73
중앙_금액,6113.72


── Payment_type


,Payment_type,거래,세탁,세탁_비율_pct
0,Credit card,2012909,1136.0,0.0564
1,Debit card,2012103,1124.0,0.0559
2,Cheque,2011419,1087.0,0.0540
3,ACH,2008807,1159.0,0.0577
4,Cross-border,933931,2628.0,0.2814
5,Cash Withdrawal,300477,1334.0,0.4440
6,Cash Deposit,225206,1405.0,0.6239


── Payment_currency


... (8 outputs left out)

,Is_laundering,Laundering_type,거래
0,1,Structuring,1870
1,1,Cash_Withdrawal,1334
2,1,Deposit-Send,945
3,1,Smurfing,932
4,1,Layered_Fan_In,656
5,1,Layered_Fan_Out,529
6,1,Stacked Bipartite,506
7,1,Behavioural_Change_1,394
8,1,Bipartite,383
9,1,Cycle,382


── 월별 거래·세탁


,월,거래,세탁
0,2022-10,708654,694.0
1,2022-11,897621,794.0
2,2022-12,900341,907.0
3,2023-01,908320,940.0
4,2023-02,904806,946.0
5,2023-03,908562,927.0
6,2023-04,902883,962.0
7,2023-05,917601,866.0
8,2023-06,897243,1024.0
9,2023-07,902239,983.0


,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type
0,10:35:19,2022-10-07,8724731955,2769355426,1459.15,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits
1,10:35:20,2022-10-07,1491989064,8401255335,6019.64,UK pounds,Dirham,UK,UAE,Cross-border,0,Normal_Fan_Out
2,10:35:20,2022-10-07,287305149,4404767002,14328.44,UK pounds,UK pounds,UK,UK,Cheque,0,Normal_Small_Fan_Out
3,10:35:21,2022-10-07,5376652437,9600420220,11895.00,UK pounds,UK pounds,UK,UK,ACH,0,Normal_Fan_In
4,10:35:21,2022-10-07,9614186178,3803336972,115.25,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits
